# SetFit — Classification de texte few-shot (émotions)

Entraînement few-shot avec **SetFit** sur le dataset `dair-ai/emotion` (6 émotions, en lien avec la psychologie).

On fait varier le **nombre d'échantillons par classe** (8, 10, 20, 50, 100) et le **nombre d'epochs** (1, 5, 10), puis on évalue chaque modèle sur le **même jeu de test** avec accuracy, precision, recall et F1.

⚠️ **Active le GPU** : *Exécution → Modifier le type d'exécution → T4 GPU*. Sinon l'entraînement des 15 modèles sera très lent.


## 1. Installation

In [ ]:
!pip install -q setfit datasets scikit-learn pandas matplotlib
print("Installé.")

## 2. Vérifier le GPU

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))

## 3. Préparer les données (split 80 / 20)

In [ ]:
from datasets import load_dataset, concatenate_datasets

DATASET = "dair-ai/emotion"
SEED = 42
TEST_SIZE = 0.20
TEST_CAP = 500

ds = load_dataset(DATASET)
full = concatenate_datasets([ds["train"], ds["validation"], ds["test"]])
split = full.train_test_split(test_size=TEST_SIZE, seed=SEED, stratify_by_column="label")
train_pool = split["train"]
test_set = split["test"].shuffle(seed=SEED).select(range(min(TEST_CAP, len(split["test"]))))

test_texts = test_set["text"]
test_labels = list(test_set["label"])
print(f"Pool d'entraînement : {len(train_pool)}  |  Test fixe : {len(test_set)}")
print("Classes :", ds["train"].features["label"].names)

## 4. Lancer la grille d'expériences (5 × 3 = 15 modèles)

Cela prend plusieurs minutes sur GPU. Patiente.

In [ ]:
import time, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset

BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # rapide
SAMPLES_PER_CLASS = [8, 10, 20, 50, 100]
EPOCHS = [1, 5, 10]
BATCH_SIZE = 16

results = []
total = len(SAMPLES_PER_CLASS) * len(EPOCHS)
i = 0
for n in SAMPLES_PER_CLASS:
    for e in EPOCHS:
        i += 1
        print(f"\n[{i}/{total}] {n} échantillons/classe, {e} epochs...")
        train_few = sample_dataset(train_pool, label_column="label", num_samples=n)
        model = SetFitModel.from_pretrained(BASE_MODEL)
        args = TrainingArguments(batch_size=BATCH_SIZE, num_epochs=e)
        trainer = Trainer(model=model, args=args, train_dataset=train_few)
        t0 = time.time()
        trainer.train()
        dt = time.time() - t0
        preds = [int(p) for p in model.predict(test_texts)]
        acc = accuracy_score(test_labels, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(test_labels, preds, average="macro", zero_division=0)
        results.append({"samples_per_class": n, "epochs": e, "train_examples": len(train_few),
                        "accuracy": round(acc,4), "precision": round(prec,4),
                        "recall": round(rec,4), "f1": round(f1,4), "train_time_s": round(dt,1)})
        print(f"   acc={acc:.3f}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}  ({dt:.0f}s)")

df = pd.DataFrame(results)
df.to_csv("results_setfit.csv", index=False)
print("\nTerminé. Résultats dans results_setfit.csv")
df

## 5. Tableau récapitulatif

In [ ]:
import pandas as pd
pd.read_csv("results_setfit.csv")

## 6. Visualisations

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv("results_setfit.csv")
pivot = df.pivot(index="samples_per_class", columns="epochs", values="f1")

fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("Epochs"); ax.set_ylabel("Échantillons par classe")
ax.set_title("F1 (macro) selon la configuration")
for y in range(pivot.shape[0]):
    for x in range(pivot.shape[1]):
        ax.text(x, y, f"{pivot.values[y,x]:.2f}", ha="center", va="center", color="white")
fig.colorbar(im, ax=ax, label="F1")
plt.tight_layout(); plt.savefig("setfit_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for e in sorted(df["epochs"].unique()):
    sub = df[df["epochs"]==e].sort_values("samples_per_class")
    ax.plot(sub["samples_per_class"], sub["f1"], marker="o", label=f"{e} epochs")
ax.set_xlabel("Échantillons par classe"); ax.set_ylabel("F1 (macro)")
ax.set_title("Effet du nombre d'échantillons sur le F1")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("setfit_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Étapes finales

1. Télécharge `results_setfit.csv`, `setfit_heatmap.png`, `setfit_curve.png`.
2. Mets le code et les résultats sur **GitHub**.
3. Envoie le lien à ton enseignant.

Pour gagner en précision (au prix de la vitesse), remplace `BASE_MODEL` par `sentence-transformers/paraphrase-mpnet-base-v2` et relance la cellule 4.
